# Où naît le flou ? — exécution sur Colab (GPU T4)

Ce notebook enchaîne les étapes du projet. **Aucune cellule n'a été exécutée par l'auteur** : le code PyTorch du dépôt n'a jamais
tourné là où il a été écrit (pas de PyTorch). L'**étape 3** (« test de fumée ») est donc obligatoire avant tout entraînement.

Avant de commencer, s'assurer d'avoir un **GPU T4** : dans le navigateur, *Exécution → Modifier le type d'exécution* ; depuis VS Code, le type se choisit à la création du serveur Colab. La cellule 2 l'affiche. Une session gratuite peut s'interrompre : tout est
**reprenable** (relancer la même cellule). Les résultats sont écrits sur Google Drive pour survivre aux coupures.

**Interface web ou VS Code ?** Les deux pilotent le même runtime et les cellules fonctionnent dans les deux. Le seul point à vérifier est **Google Drive** : son montage demande une autorisation à la page du notebook et peut échouer depuis VS Code (`MessageError: [dfs_ephemeral] ... unauthorized`). La cellule 2 le détecte et affiche `persistant : True` ou `False`. Si c'est `False`, les `runs/` vivent sur le disque de la session et disparaissent avec elle : lancer alors la campagne (étapes 7 et 9) depuis le navigateur. Si c'est `True`, VS Code convient pour tout.


## 1. GPU, dépôt et Drive

In [3]:
!nvidia-smi -L || echo "AUCUN GPU : Exécution -> Modifier le type d'exécution -> T4 (obligatoire pour les étapes 3, 7 et 9)"

# Le montage de Drive demande une autorisation à la page du notebook : il peut échouer depuis VS Code
# (erreur « dfs_ephemeral ... unauthorized »). On bascule alors sur le disque de la session, en le disant.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, PERSISTANT = '/content/drive/MyDrive/flou_wm', True
except Exception as e:
    BASE, PERSISTANT = '/content/flou_wm_resultats', False
    print(f"Drive indisponible ({type(e).__name__}). Les résultats iront dans {BASE},")
    print("qui DISPARAÎT à la fin de la session : entraînements non reprenables.")
    print("Pour la campagne complète, ouvrir ce notebook dans l'interface web de Colab.")
print(f"résultats -> {BASE}   (persistant : {PERSISTANT})")

GPU 0: Tesla T4 (UUID: GPU-db8925ca-dc7e-0297-8216-595079bae897)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
résultats -> /content/drive/MyDrive/flou_wm   (persistant : True)


In [4]:
# Cloner le dépôt (dépôt privé : mettre un jeton -> https://<TOKEN>@github.com/...)
import os, shutil
REPO = 'https://github.com/CedricDada/flou-world-models.git'
# Clone neuf à chaque exécution (quelques secondes) : c'est la seule façon sûre d'avoir le dernier code.
# Un `git pull` échouerait ici, car data/ runs/ reports/ ont été remplacés par des liens vers Drive.
# Rien n'est perdu : rmtree supprime les liens, pas leur contenu sur Drive.
if os.path.isdir('/content/flou_wm'):
    %cd /content
    shutil.rmtree('/content/flou_wm')
!git clone -q $REPO /content/flou_wm
%cd /content/flou_wm
!git log --oneline -1

# runs/ reports/ data/ pointent vers BASE (Drive si disponible) : reprise possible après une coupure
import shutil
for d in ('runs', 'reports', 'data'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        shutil.rmtree(d)
    if not os.path.islink(d):
        os.symlink(f'{BASE}/{d}', d)

# Colab fournit déjà numpy, scipy, matplotlib et pillow : n'installer que ce qui manque,
# pour ne pas risquer de remplacer la numpy sur laquelle le PyTorch préinstallé est compilé.
import importlib.util
manquants = [pkg for mod, pkg in (('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib'),
                                  ('PIL', 'pillow'), ('pytest', 'pytest'))
             if importlib.util.find_spec(mod) is None]
if manquants:
    !pip install -q {' '.join(manquants)}
print('paquets installés :', ' '.join(manquants) if manquants else 'aucun, tout était présent')

/content/flou_wm
be1f64f (HEAD -> main, origin/main, origin/HEAD) Surapprentissage corrige (400 000 transitions), relecture externe appliquee, journal de bord
paquets installés : aucun, tout était présent


## 2. Tests NumPy (jeu, bots, instruments de mesure, analyse)

Ils ne demandent pas PyTorch. Une cellule suffit ; attendre environ 2 à 4 minutes.

In [5]:
!python -m pytest -q -x

........................................................................ [ 27%]
........................................................................ [ 54%]
....................................................................ssss [ 81%]
s..............................s.................                        [100%]
=============================== warnings summary ===============================
tests/test_env.py::test_gymnasium_spaces_when_installed
  /usr/local/lib/python3.13/dist-packages/gymnasium/utils/env_checker.py:496: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
    logger.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html


## 3. Test de fumée du code PyTorch (obligatoire, une fois)

Première exécution du code PyTorch. Le script **fabrique ses propres petites données** dans un dossier temporaire (il ne dépend donc
pas de l'étape 4) et vérifie, pour chaque famille : le nombre de paramètres, un entraînement de 6 pas en lot 8, la reprise depuis un
checkpoint, l'évaluation complète, et l'extension « décision » (niveau 1 puis planificateur, 2 parties de 2 pas). Avec `--time`, il
mesure en plus le temps par pas **à la taille réelle** (lot 64) et le débit d'inférence, puis écrit `reports/timing.json`, lu à
l'étape 6.

Rien n'est écrit dans `runs/` : le dossier temporaire est supprimé à la fin. Les vérifications sont indépendantes, donc **une seule
exécution révèle toutes les erreurs** — inutile de corriger une à une. Si une ligne est `[FAIL]`, **ne pas lancer le plan** : copier
la trace et la corriger d'abord. Ce test ne dit pas que les modèles apprennent bien, seulement que la plomberie tient.

In [7]:
!python scripts/smoke_test.py --time

PyTorch 2.14.1+cu130 sur Tesla T4
[ OK ] paramètres regression — 5 872 771 paramètres (= plan)  (0.1 s)
[ OK ] paramètres oracle — 5 873 635 paramètres (= plan)  (0.1 s)
[ OK ] paramètres diffusion — 6 005 651 paramètres (= plan)  (0.1 s)
[ OK ] paramètres tokens — 6 043 715 paramètres (= plan)  (0.1 s)
       écart de taille entre familles : 2.9 % (budget apparié visé : quelques %)
[ OK ] sonde CNN (forme de sortie) — 2 525 896 paramètres  (1.6 s)
données de fumée : 2000 transitions d'entraînement, 1000 de validation, 60 états de test
/content/flou_wm/src/flou/train/trainer.py:183: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  row = {"stage": st.name, "step": n, "loss": float(loss), "lr": lr, "grad_norm": float(gnorm),
[ OK ] entraînement (6 pas) regression — 6 journaux, perte 0.671

## 4. Données

Les valeurs de p du plan (0,1 / 0,3 / 0,5) plus p = 0 (contrôles), en entraînement, validation et test. Générées une fois, sur Drive.

In [6]:
!python scripts/generate_data.py --split train --p 0.0 0.1 0.3 0.5 --n-transitions 400000
!python scripts/generate_data.py --split val   --p 0.0 0.1 0.3 0.5 --n-transitions 20000
!python scripts/generate_data.py --split test  --p 0.0 0.1 0.3 0.5 --n-bifurcations 2000   # H2b exige >= 1 000 images NETTES par graine

p=0.00  data/p0.00/train_random.npz  (1.3 Mo, 4.4 s)
p=0.00  data/p0.00/train_greedy.npz  (1.3 Mo, 4.4 s)
p=0.10  data/p0.10/train_random.npz  (1.3 Mo, 3.9 s)
p=0.10  data/p0.10/train_greedy.npz  (1.4 Mo, 3.9 s)
p=0.30  data/p0.30/train_random.npz  (1.3 Mo, 3.9 s)
p=0.30  data/p0.30/train_greedy.npz  (1.4 Mo, 3.9 s)
p=0.50  data/p0.50/train_random.npz  (1.3 Mo, 4.9 s)
p=0.50  data/p0.50/train_greedy.npz  (1.4 Mo, 4.9 s)
p=0.00  data/p0.00/val_random.npz  (0.1 Mo, 1.5 s)
p=0.00  data/p0.00/val_greedy.npz  (0.1 Mo, 1.5 s)
p=0.10  data/p0.10/val_random.npz  (0.1 Mo, 1.4 s)
p=0.10  data/p0.10/val_greedy.npz  (0.1 Mo, 1.4 s)
p=0.30  data/p0.30/val_random.npz  (0.1 Mo, 1.4 s)
p=0.30  data/p0.30/val_greedy.npz  (0.1 Mo, 1.4 s)
p=0.50  data/p0.50/val_random.npz  (0.1 Mo, 1.4 s)
p=0.50  data/p0.50/val_greedy.npz  (0.1 Mo, 1.4 s)
p=0.00  data/p0.00/test_bifurcation.npz  (0.1 Mo, 0.8 s)
p=0.10  data/p0.10/test_bifurcation.npz  (0.1 Mo, 0.8 s)
p=0.30  data/p0.30/test_bifurcation.npz  (0.1 Mo, 0.8 

In [20]:
import shutil, os, glob
os.makedirs(f'{BASE}/archive', exist_ok=True)
shutil.move('runs/core-regression-p0.30-s0', f'{BASE}/archive/core-regression-p0.30-s0_N100k')
for d in glob.glob('runs/core-*') + glob.glob('runs/control-*'):
    shutil.rmtree(d)
if os.path.exists('runs/plan_status.json'):
    os.remove('runs/plan_status.json')
print('reste :', sorted(os.listdir('runs')))


reste : []


## 5. Instrument de mesure (avant tout modèle)

Règle ρmax et contrôle H0a sans entraînement. Si H0a échoue, **ne mesurer aucun modèle**.

In [7]:
!python scripts/validate_instrument.py

8000 paires (A, B) sur 4 jeux de test ; δ dans [6.71, 7.65]
ρmax = 0.1599  (bruit synthétique σ = 0.01, quantile 0.995)
H0a (contrôle de l'instrument, sans entraînement) :
{
 "rho_max": 0.15988816883972015,
 "alpha_mae_mix": 0.001048432968625682,
 "pure_a_ok": 0.994875,
 "pure_b_ok": 0.994625,
 "rho_pure_q995": 0.159971278492099,
 "rho_mix_max": 0.1667269972912107,
 "pure_ok": 0.99475,
 "mix_not_other": 0.99625,
 "broken_flagged_other": 1.0,
 "rho_broken_min": 0.35206978138219175,
 "noise_sigma": 0.01,
 "pass_alpha": true,
 "pass_pure": true,
 "pass": true
}
sensibilité au seuil :
   x0.5 : ρmax = 0.0799  {"pure_ok": 0.0, "alpha_mae_mix": 0.001048432968625682, "broken_flagged_other": 1.0, "pass": false}
     x1 : ρmax = 0.1599  {"pure_ok": 0.99475, "alpha_mae_mix": 0.001048432968625682, "broken_flagged_other": 1.0, "pass": true}
     x2 : ρmax = 0.3198  {"pure_ok": 1.0, "alpha_mae_mix": 0.001048432968625682, "broken_flagged_other": 1.0, "pass": true}
rapport : reports/instrument.json


## 6. Plan et budget

Le nombre de pas par run se règle sur le temps **mesuré** à l'étape 3 (pas sur une valeur supposée).

In [10]:
!python scripts/plan.py --timing reports/timing.json
# Pour caler le plan sur un budget : par exemple 38 h de GPU
!python scripts/plan.py --timing reports/timing.json --budget-hours 38

 #  run                                groupe      p graine   params      pas  images vues  note
 1  core-regression-p0.30-s0           core     0.30      0   5.87 M    20000    1 280 000  
 2  core-diffusion-p0.30-s0            core     0.30      0   6.01 M    20000    1 280 000  
 3  core-tokens-K256-p0.30-s0          core     0.30      0   6.04 M    20000    1 280 000  
 4  core-regression-p0.30-s1           core     0.30      1   5.87 M    20000    1 280 000  
 5  core-diffusion-p0.30-s1            core     0.30      1   6.01 M    20000    1 280 000  
 6  core-tokens-K256-p0.30-s1          core     0.30      1   6.04 M    20000    1 280 000  
 7  core-regression-p0.30-s2           core     0.30      2   5.87 M    20000    1 280 000  
 8  core-diffusion-p0.30-s2            core     0.30      2   6.01 M    20000    1 280 000  
 9  core-tokens-K256-p0.30-s2          core     0.30      2   6.04 M    20000    1 280 000  
10  control-oracle-p0.30-s0            control  0.30      0   5.87

## 7. Exécution

Cœur et contrôles d'abord. Remplacer `--steps` par la valeur calculée à l'étape 6.

`--max-hours 10` est un **arrêt propre**, pas une interruption : le run en cours reçoit une limite de temps, termine son pas, écrit
son checkpoint, et `runs/plan_status.json` garde l'état du plan. Relancer la même cellule reprend exactement où l'on s'était arrêté,
sans refaire les runs terminés. Sur une session gratuite il protégeait de la coupure ; avec Colab Pro (jusqu'à 24 h) il sert surtout
de garde-fou de consommation — regarder le compteur d'unités de calcul entre deux relances.


In [22]:
!python scripts/run_plan.py --steps 34800 --groups core control --dry

[ 1/12] core-regression-p0.30-s0           à faire
[ 2/12] core-diffusion-p0.30-s0            à faire
[ 3/12] core-tokens-K256-p0.30-s0          à faire
[ 4/12] core-regression-p0.30-s1           à faire
[ 5/12] core-diffusion-p0.30-s1            à faire
[ 6/12] core-tokens-K256-p0.30-s1          à faire
[ 7/12] core-regression-p0.30-s2           à faire
[ 8/12] core-diffusion-p0.30-s2            à faire
[ 9/12] core-tokens-K256-p0.30-s2          à faire
[10/12] control-oracle-p0.30-s0            à faire
[11/12] control-regression-p0.00-s0        à faire
[12/12] control-diffusion-p0.00-s0         à faire


In [23]:
import json, os
for f in ('timing.json', 'instrument.json'):
    print(f, ':', os.path.exists(f'reports/{f}'))
if os.path.exists('reports/instrument.json'):
    h = json.load(open('reports/instrument.json'))['h0a']
    print('H0a :', {k: h[k] for k in h if not isinstance(h[k], (list, dict))})
if os.path.exists('reports/timing.json'):
    print('timing :', json.load(open('reports/timing.json')))


timing.json : True
instrument.json : True
H0a : {'rho_max': 0.1613945950079933, 'alpha_mae_mix': 0.0010423491208280291, 'pure_a_ok': 0.995, 'pure_b_ok': 0.9945, 'rho_pure_q995': 0.16192693666915184, 'rho_mix_max': 0.16671661957154393, 'pure_ok': 0.99475, 'mix_not_other': 0.99525, 'broken_flagged_other': 1.0, 'rho_broken_min': 0.3518661371652065, 'noise_sigma': 0.01, 'pass_alpha': True, 'pass_pure': True, 'pass': True}
timing : {'device': 'cuda', 'gpu': 'Tesla T4', 'torch': '2.14.1+cu130', 'batch_size': 64, 'amp': True, 'time_steps': 15, 'seconds_per_step': {'regression': 0.2331608772277832, 'oracle': 0.23480820655822754, 'diffusion': 0.23882654507954915, 'tokens-vq': 0.09372442563374837, 'tokens-gpt': 0.07119986216227213}, 'peak_memory_gb': {'regression': 3.677217792, 'oracle': 3.6706816, 'diffusion': 3.67551744, 'tokens-vq': 1.512692224, 'tokens-gpt': 1.208944128}, 'inference_images_per_second': {'regression': 714.5158727430055, 'diffusion': 242.37153783682706, 'tokens': 42.8815036618

In [ ]:
#!python scripts/run_plan.py --steps 34800 --max-hours 10 --groups core control
!python scripts/train.py --run-id core-regression-p0.30-s0 --steps 34800
!python scripts/evaluate.py --run-id core-regression-p0.30-s0


core-regression-p0.30-s0 : 34800 pas, lot 64, p = 0.3, graine 0
reprise : stade 0, pas 10000, 0.77 h déjà consommées
[main] 10100/34800  perte 0.0010  lr 1.67e-04
[main] 10200/34800  perte 0.0005  lr 1.67e-04
[main] 10300/34800  perte 0.0013  lr 1.66e-04
[main] 10400/34800  perte 0.0012  lr 1.65e-04
[main] 10500/34800  perte 0.0006  lr 1.65e-04
[main] 10600/34800  perte 0.0009  lr 1.64e-04
[main] 10700/34800  perte 0.0004  lr 1.64e-04
[main] 10800/34800  perte 0.0011  lr 1.63e-04
[main] 10900/34800  perte 0.0007  lr 1.62e-04
[main] 11000/34800  perte 0.0013  lr 1.61e-04
[main] 11100/34800  perte 0.0007  lr 1.61e-04
[main] 11200/34800  perte 0.0014  lr 1.60e-04
[main] 11300/34800  perte 0.0014  lr 1.59e-04
[main] 11400/34800  perte 0.0003  lr 1.59e-04
[main] 11500/34800  perte 0.0013  lr 1.58e-04
[main] 11600/34800  perte 0.0007  lr 1.57e-04
[main] 11700/34800  perte 0.0007  lr 1.57e-04
[main] 11800/34800  perte 0.0007  lr 1.56e-04
[main] 11900/34800  perte 0.0007  lr 1.55e-04
[main] 12

In [ ]:
# Courbes d'entraînement + alerte de surapprentissage (voir docs/JOURNAL.md, J6 et J7).
# Code de sortie non nul si la validation remonte ou si la perte passe sous le plancher de Bayes.
!python scripts/training_curves.py --all

from IPython.display import Image, display
import glob
for f in sorted(glob.glob('runs/*/fig_training.png')):
    print(f)
    display(Image(f))

In [ ]:
!python scripts/run_plan.py --steps 34800 --max-hours 10 --skip-groups core control p_axis

In [ ]:
!python scripts/run_plan.py --steps 34800 --max-hours 10 --groups p_axis    # exploratoire, sacrifié en premier

## 8. Analyse

Verdicts H0 à H3 (H4 et H5 restent « non évaluables » tant que l'étape 9 n'est pas faite), tables, figures, rapport. Puis, à la main : la relecture de 100 cas et le test A/B (voir `docs/EXPERIMENTS.md`).

In [ ]:
!python scripts/analyze.py

In [ ]:
import glob
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/report.md', encoding='utf-8').read()))
# toutes les figures produites : flou, flou selon p, calibration, rappel du petit objet,
# MSE contre plancher de Bayes, suivi en rollout
for f in sorted(glob.glob('reports/fig_*.png')):
    print(f)
    display(Image(f))

### Bandes d'images : ce que le modèle imagine

Illustrations seulement, aucun verdict n'en dépend. Pour chaque état de bifurcation, deux lignes : la réalité, puis les images
que le modèle prédit en autorégressif (il se nourrit de ses propres sorties). C'est là que le flou se voit — la régression se
brouille pas après pas, un échantillonneur reste net mais peut dériver. Écrit `runs/<run>/filmstrip_*.png`, et des GIF
`réel | imaginé` avec `--gif`. À lancer après l'étape 7, sur les runs terminés.

In [ ]:
!python scripts/filmstrip.py --run-id core-regression-p0.30-s0 --gif
!python scripts/filmstrip.py --run-id core-diffusion-p0.30-s0 --n-steps 3 --gif

from IPython.display import Image, display
for f in sorted(glob.glob('runs/core-*/filmstrip_*.png')):
    print(f)
    display(Image(f))

## 9. Extension « décision » (facultatif) : le flou change-t-il une décision ?

Hypothèses H4 et H5, guidées par PETS (arXiv:1805.12114), PlaNet (arXiv:1811.04551) et Objective Mismatch (arXiv:2002.04523) : voir `docs/EXTENSION_DECISION.md`.
D'abord les contrôles sans modèle (H0e, H0f ; quelques minutes de CPU), puis le niveau 1, rapide.

In [ ]:
# H0f aux réglages du niveau 2 (DECISIONS.md, C7 révisé) : 32 suites, 20 parties
!python scripts/validate_decision.py --games 20 --candidates 32
!python scripts/evaluate_decision.py --all

Niveau 2 (planification dans le modèle) : **long**. La première cellule chronomètre sur 4 parties de 10 pas ; l'étape 3 (`smoke_test.py --time`) a aussi affiché une estimation.
Si c'est trop long, réduire `--games` ou `--candidates` **pour tous les runs**. Relancer la même cellule après une coupure : les runs déjà faits sont sautés.

In [ ]:
!python scripts/evaluate_decision.py --all --level 2 --games 4 --game-steps 10 --no-save   # chronométrage seulement, n'écrit rien

In [ ]:
!python scripts/evaluate_decision.py --all --level 2 --games 20 --candidates 32   # réglages réduits (voir docs/DECISIONS.md, C7) ; reprenable


In [ ]:
!python scripts/analyze.py   # le rapport contient maintenant H4 et H5